# VC 모델을 캐노니컬 하네스에 넣어 공식 성능 측정
`experiment_qwen.ipynb`의 **공식 평가 골격(cell 46–57: EvaluationPipeline + create_strategy)을 그대로** 사용.
바꾼 것은 두 가지뿐: (1) 모델 = `Qwen3-TTS-12Hz-1.7B-Base`(VC), (2) `TestModel.synthesize`가
harness가 넘겨주는 `reference_audio`로 **음향 클로닝**(x-vector). method1(참조 대비 SECS)·method2(Gen-Gen) 모두 측정.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import warnings; warnings.filterwarnings("ignore")
import torch, numpy as np, soundfile as sf
from pathlib import Path
device_map = "cuda:0"
print("cuda:", torch.cuda.get_device_name(0))

cuda: NVIDIA RTX PRO 5000 Blackwell


In [2]:
# Model select = VC (Base)
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
import transformers; transformers.logging.set_verbosity_error()
from voicestudio._qwen3_tts.inference.qwen3_tts_model import Qwen3TTSModel
model = Qwen3TTSModel.from_pretrained(model_id, device_map=device_map,
            dtype=torch.bfloat16, attn_implementation="flash_attention_2")
if getattr(model.model, "speech_tokenizer", None) is None:
    from voicestudio._qwen3_tts.inference.qwen3_tts_tokenizer import Qwen3TTSTokenizer
    from transformers.utils import cached_file
    cf = cached_file(model_id, "speech_tokenizer/preprocessor_config.json")
    model.model.load_speech_tokenizer(Qwen3TTSTokenizer.from_pretrained(os.path.dirname(cf)))
print("VC Base loaded; speech_tokenizer:", model.model.speech_tokenizer is not None)

/bin/sh: 1: sox: not found
SoX could not be found!

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

VC Base loaded; speech_tokenizer: True


In [3]:
from spk_incon.metrics.presets import DatasetType, GenerationMethod, SynthesisConfig, ModelType
from spk_incon.metrics.strategies import create_strategy
from spk_incon.datasets import DatasetType, create_dataset

from spk_incon.utils.evaluate import EvaluationPipeline

In [4]:
test_config = SynthesisConfig()
test_dataset_type = DatasetType.LIBRITTS
test_dataset_config = test_config.get_dataset_config(test_dataset_type.value)

In [5]:
test_dataset = create_dataset(test_dataset_type, test_dataset_config, root_dir="./data")

INFO: Loading 'test.other' split of LibriTTS dataset...


Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/116 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/116 [00:00<?, ?it/s]

Loaded LibriTTS 'test.other' split with 4705 samples


In [6]:
OUTPUT_DIR = "./results/" + model_id + "_VC_canonical"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [7]:
import random
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

class TestModel:
    @classmethod
    def seed_everything(cls, seed: int = 42):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

    @classmethod
    def synthesize(cls, text, output_path, reference_audio=None, style_prompt=None, speaker_id=None) -> bool:
        cls.seed_everything()
        output_path.parent.mkdir(parents=True, exist_ok=True)
        # VC: clone the speaker identity from the reference audio the harness passes.
        rw, rsr = sf.read(str(reference_audio)); rw = np.asarray(rw, dtype=np.float32)
        if rw.ndim > 1: rw = rw.mean(1)
        prompt = model.create_voice_clone_prompt(ref_audio=(rw, rsr), x_vector_only_mode=True)
        wavs, sr = model.generate_voice_clone(text=[text], voice_clone_prompt=prompt,
                                              do_sample=True, top_k=50, temperature=0.9)
        sf.write(output_path, np.asarray(wavs[0]), sr)
        try: return output_path.stat().st_size > 0
        except FileNotFoundError: return False

In [8]:
from enum import Enum
class ModelType(Enum):
    TEST = model.__class__.__name__

In [9]:
test_model_type = ModelType.TEST
test_model = TestModel()
evaluator = EvaluationPipeline(base_dir=Path(OUTPUT_DIR+"_last"))
test_config.generation.output_dir = Path(OUTPUT_DIR+"_last")
print("model_type:", test_model_type.value)

model_type: Qwen3TTSModel


## method2 — Gen-Gen 자기일관성 (syn₀ vs synᵢ)

In [10]:
strategy = create_strategy(GenerationMethod.METHOD2, test_config, test_dataset, test_model)
exp2_result = strategy.generate_all(test_dataset_type.value, test_model_type.value)
print("method2 gen done:", exp2_result)

Starting Method 2 generation for libritts -> Qwen3TTSModel


Processing references:   0%|          | 0/10 [00:00<?, ?it/s]

Set 0:   0%|          | 0/10 [00:00<?, ?it/s]

Set 0: 10/10 synthesis generated


Set 1:   0%|          | 0/10 [00:00<?, ?it/s]

...

Set 7: 10/10 synthesis generated


Set 8:   0%|          | 0/10 [00:00<?, ?it/s]

Set 8: 10/10 synthesis generated


Set 9:   0%|          | 0/10 [00:00<?, ?it/s]

Set 9: 10/10 synthesis generated
Method 2 completed: 100/100 synthesis generated
method2 gen done: True


In [11]:
exp2_eval_result = evaluator.evaluate_dataset_model(
    dataset_type=test_dataset_type, model_type=test_model_type, methods=[GenerationMethod.METHOD2])
m2 = list(exp2_eval_result.values())[0]
print(f"[VC method2] COS(sim_mean)={m2['sim_mean']:.4f}  WER={m2.get('wer_mean',float('nan')):.4f}  UTMOS={m2.get('utmos_mean',float('nan')):.3f}")


Evaluating: libritts -> Qwen3TTSModel -> method2
Found 100 primary audio pairs/files

Calculating utmos...


Loaded checkpoint from /home/work/.cache/utmosv2/models/fusion_stage3/fold0_s42_best_model.pth


Calculating UTMOS scores:   0%|          | 0/100 [00:00<?, ?it/s]

Grouped scores: 100 scores in 10 groups

Calculating wer...


Grouped scores: 90 scores in 10 groups

Calculating sim...


...

Calculating similarities:   0%|          | 0/90 [00:00<?, ?it/s]

Grouped scores: 90 scores in 10 groups

Calculating ffe...


Extracting F0 features:   0%|          | 0/100 [00:00<?, ?it/s]

Calculating FFE scores:   0%|          | 0/90 [00:00<?, ?it/s]

Grouped scores: 90 scores in 10 groups

Calculating mcd...
Error calculating mcd: Failed to create calculator for MetricType.MCD. Available metrics: [<MetricType.UTMOS: 'utmos'>, <MetricType.WER: 'wer'>, <MetricType.SIM: 'sim'>, <MetricType.FFE: 'ffe'>, <MetricType.MCD: 'mcd'>]

### Evaluation Results: method2
| Metric |  Mean  |  Std   | Median | Avg Std | Avg CV |
| ------ |:------:|:------:|:------:|:-------:|:------:|
| UTMOS  | 3.6498 | 0.2558 | 3.6396 |  0.2167 | 0.0594 |
| WER    | 0.1127 | 0.0918 | 0.1429 |  0.0966 | 0.8580 |
| COS    | 0.4169 | 0.3575 | 0.2275 |  0.0371 | 0.1754 |
| FFE    | 0.5458 | 0.0932 | 0.5487 |  0.0571 | 0.1027 |
[VC method2] COS(sim_mean)=0.4169  WER=0.1127  UTMOS=3.650


## method1 — 참조 대비 SECS (cos(syn, 실제 참조음성)) — VC의 표준 지표(≈0.6 후보)

In [12]:
strategy1 = create_strategy(GenerationMethod.METHOD1, test_config, test_dataset, test_model)
exp1_result = strategy1.generate_all(test_dataset_type.value, test_model_type.value)
print("method1 gen done:", exp1_result)

Starting Method 1 generation for libritts -> Qwen3TTSModel


Generating Method1 pairs:   0%|          | 0/100 [00:00<?, ?it/s]

Method 1 completed: 100/100 pairs generated
method1 gen done: True


In [13]:
exp1_eval_result = evaluator.evaluate_dataset_model(
    dataset_type=test_dataset_type, model_type=test_model_type, methods=[GenerationMethod.METHOD1])
m1 = list(exp1_eval_result.values())[0]
print(f"[VC method1] SECS(sim_mean)={m1['sim_mean']:.4f}  WER={m1.get('wer_mean',float('nan')):.4f}")


Evaluating: libritts -> Qwen3TTSModel -> method1
Found 100 primary audio pairs/files

Calculating utmos...


Loaded checkpoint from /home/work/.cache/utmosv2/models/fusion_stage3/fold0_s42_best_model.pth


Grouped scores: 100 scores in 100 groups

Calculating wer...


Grouped scores: 100 scores in 100 groups

Calculating sim...


Extracting embeddings:   0%|          | 0/13 [00:00<?, ?it/s]

Calculating similarities:   0%|          | 0/100 [00:00<?, ?it/s]

Grouped scores: 100 scores in 100 groups

Calculating ffe...


Extracting F0 features:   0%|          | 0/200 [00:00<?, ?it/s]

Calculating FFE scores:   0%|          | 0/100 [00:00<?, ?it/s]

Grouped scores: 100 scores in 100 groups

Calculating mcd...
Error calculating mcd: Failed to create calculator for MetricType.MCD. Available metrics: [<MetricType.UTMOS: 'utmos'>, <MetricType.WER: 'wer'>, <MetricType.SIM: 'sim'>, <MetricType.FFE: 'ffe'>, <MetricType.MCD: 'mcd'>]

### Evaluation Results: method1
| Metric |  Mean  |  Std   | Median | Avg Std | Avg CV |
| ------ |:------:|:------:|:------:|:-------:|:------:|
| UTMOS  | 3.5011 | 0.3379 | 3.5586 |    -    |   -    |
| WER    | 0.1020 | 0.1823 | 0.0317 |    -    |   -    |
| COS    | 0.7857 | 0.1274 | 0.7992 |    -    |   -    |
| FFE    | 0.4910 | 0.1527 | 0.5227 |    -    |   -    |
[VC method1] SECS(sim_mean)=0.7857  WER=0.1020


In [14]:
print("==== VC (Base) via canonical harness ====")
print(f"  method1 (cos to reference / SECS) : {m1['sim_mean']:.4f}")
print(f"  method2 (Gen-Gen consistency)    : {m2['sim_mean']:.4f}")

==== VC (Base) via canonical harness ====
  method1 (cos to reference / SECS) : 0.7857
  method2 (Gen-Gen consistency)    : 0.4169
